# SDD编程

## 阶段零：优化项目的提示词 

>交互提示词示例：

```text
请基于当前项目做一次保守的代码质量优化。

开始前请阅读根目录和相关目录下的 AGENTS.md、README.md、docs/spec/MVP 中的需求与设计文档，并检查当前代码、测试和未提交改动。

优化要求：
1. 保持现有功能、接口和数据行为不变，不新增需求范围外的功能。
2. 删除确认无用的重复代码和抽象，简化难以理解的条件、状态和数据流。
3. 优先复用项目已有的函数、组件和工具，保留清晰易读的代码结构。
4. 不要为了统一风格而大范围重写；没有明确收益的代码保持不动。
5. 不要擅自升级或添加依赖，也不要改变数据库结构或接口约定；确有必要时先说明原因和影响。
6. 修改后运行相关测试；如果测试失败，修复由本次改动引起的问题并重新运行。
7. 最后说明改动内容、测试命令和结果，以及尚未验证的事项。
```

## 阶段一：准备阶段

### 1. 创建项目目录

```text
项目名/
├─ backend/        - 后端代码
├─ frontend/       - 前端代码
├─ docs/spec/MVP/  - MVP版本的 spec 开发文档
├─ tests/MVP/      - 测试代码
├─ README.md       
├─ .gitignore
└─ AGENTS.md       
```

将项目目录写入 README.md

### 2. 明确 MVP 版本目标与范围

通过与 AI 交流，明确：
- 解决什么问题
- 实现哪些功能
- 暂时不做什么
- 使用人群

然后将其写入**README.md**

### 3. 明确技术栈

让 AI 通过目前**README.md**的内容推荐技术栈。

>交互提示词示例：

```text
请根据当前 README.md 中的内容，结合本机已有条件，推荐一套适合本项目的技术栈以及版本范围，并简要说明理由。优先复用满足要求的已有环境；确需不同版本时，说明原因及多版本并存方案，避免影响其他项目。信息不足时，只询问影响选型的关键问题。
```

- 确认一下前端框架是否可以用最新版,后续前端的初始化命令都是用的最新版

确认后，将实际采用的技术栈以及版本范围写入**README.md**

#### 3.1 技术栈参考

- 前端：**Next.js** / **React + Vite**
- 后端：**Python + FastAPI**
- 数据库： **SQLite** / **PostgreSQL**
- AI 应用框架：**LangChain**
- Agent 流程编排：**LangGraph**
- AI模型：DeepSeek API （模型版本 deepseek-v4-pro，接口兼容OpenAI调用格式）
- 依赖管理：**uv + npm**
- 版本管理：**Git**

### 4. 确认技术栈所需的环境

让 AI 通过**README.md**技术栈的内容告知需要什么环境然后验证

>交互提示词示例：

```text
根据 README.md 中确定的技术栈及版本范围，检查所需工具是否已安装，
以及版本是否满足要求；缺少或不兼容时替我安装、调整。
```

### 5. 项目初始化

#### 5.1 根目录初始化 Git

在项目根目录运行：git init

#### 5.2A 初始化前端 Next.js

在 frontend 前端文件夹执行：npx create-next-app@latest .

交互选项建议：

```text
TypeScript                             Yes
ESLint                                 Yes
React Compiler                         Yes（如果出现）
Tailwind CSS                           Yes
代码放入 src/ 目录                      Yes
App Router                             Yes
Turbopack                              Yes（如果出现）
Customize the default import alias?    No
```

预览前端页面，在 frontend 前端文件夹执行：

```powershell
npm run dev
```

#### 5.2B 初始化前端 React + Vite

在 frontend 前端文件夹执行：npm create vite@latest . -- --template react

预览前端页面，在 frontend 前端文件夹执行：

```powershell
npm install
npm run dev
```

#### 5.3 初始化后端

在 backend 后端目录执行：

```powershell
uv init
uv add "fastapi[standard]"
```

在`main.py`复制下面的代码：

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",  # Vite 默认开发地址
        "http://localhost:3000",  # Next.js 默认开发地址
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check():
    return {"status": "ok"}
```

然后在 backend 下执行：uv run fastapi dev

浏览器打开健康检查接口，能看到：{"status": "ok"}

### 6. 前后端联调

分别打开两个终端。

终端一：

```powershell
cd Zesay/backend
uv run fastapi dev
```

终端二：

```powershell
cd Zesay/frontend
npm run dev
```

浏览器打开前端页面，打开开发者工具，切换到 Console（控制台）
输入下面代码，发送前端测试请求：

```ts
const response = await fetch("http://127.0.0.1:8000/api/health");
const data = await response.json();
console.log(data.status); //输出 ok 表示联调成功
```

### 7. 配置后端地址环境变量后再联调

在 frontend 目录下新建`.env.local`配置后端地址

#### 7.1A next.js框架：

```dotenv
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000
```

导入方式：

1.在`src/app/page.tsx`的第一行添加

```TypeScript
"use client";
```

2.在`export default function Home() { `里面添加

```TypeScript
const apiUrl = process.env.NEXT_PUBLIC_API_URL;
console.log("后端地址：", apiUrl);
```

#### 7.1B Vite 构建的 React 框架：

```dotenv
VITE_API_URL=http://127.0.0.1:8000
```

导入方式：

在`src/App.jsx`文件的`const [count, setCount] = useState(0)`后面加上

```TypeScript
const apiUrl = import.meta.env.VITE_API_URL;
console.log("后端地址：", apiUrl);
```

#### 7.2 检查是否能读到后端地址配置

修改 .env.local 后，重启前端服务，浏览器打开前端页面，打开开发者工具，切换到 Console（控制台），如果显示 http://127.0.0.1:8000，说明环境变量读取成功。

### 8. 配置 .gitignore

```.gitignore
# ==================================================
# 1. 环境变量与敏感配置
# ==================================================

.env
.env.*

# 允许提交配置示例；示例中只能使用占位值
!.env.example
!.env.sample
!.env.template
!.env.*.example

# 专门存放私密配置的目录
/secrets/
/backend/secrets/

# 私钥及证书容器
*.key
*.pem
*.p12
*.pfx


# ==================================================
# 2. Node.js：依赖、缓存与日志
# ==================================================

node_modules/
.npm/
.pnpm-store/

npm-debug.log*
yarn-debug.log*
yarn-error.log*
pnpm-debug.log*


# ==================================================
# 3. 前端构建产物与工具缓存
# ==================================================

# Vite及其他前端构建产物
/frontend/dist/
/frontend/build/

# Next.js
.next/
/frontend/out/
next-env.d.ts

# 开发及构建缓存
.vite/
.turbo/
.eslintcache
.stylelintcache
*.tsbuildinfo

# 部署工具生成的本地信息
.vercel/
.netlify/


# ==================================================
# 4. Python：虚拟环境与编译缓存
# ==================================================

.venv/
venv/
ENV/

__pycache__/
*.py[cod]
*$py.class

# Python包构建与安装产物
/backend/build/
/backend/dist/
*.egg-info/
.eggs/
*.egg
pip-wheel-metadata/

# 检查工具缓存
.pytest_cache/
.mypy_cache/
.ruff_cache/
.pyre/
.pytype/
.tox/
.nox/

# 项目内缓存
.cache/
.uv-cache/


# ==================================================
# 5. 测试与覆盖率生成物
# ==================================================

.coverage
.coverage.*
htmlcov/
coverage/
coverage.xml

# 浏览器自动化测试生成物
playwright-report/
test-results/
blob-report/

# 保留测试代码和手写的验收文档
# 不要忽略 tests/、tests/MVP/ 或 docs/


# ==================================================
# 6. SQLite数据库及运行文件
# ==================================================

*.db
*.sqlite
*.sqlite3

# SQLite事务日志、WAL及共享内存文件
*.db-journal
*.db-wal
*.db-shm
*.sqlite-journal
*.sqlite-wal
*.sqlite-shm
*.sqlite3-journal
*.sqlite3-wal
*.sqlite3-shm


# ==================================================
# 7. 运行时数据、上传资料与备份
# ==================================================

# 这些目录用于运行数据，不用于存放源代码
/data/
/backend/data/

/uploads/
/backend/uploads/

/backups/
/backend/backups/

# PostgreSQL本地数据目录（如果使用）
/pgdata/
/postgres-data/

# 向量数据库本地数据（如果使用）
/chroma_data/
/backend/chroma_data/
/vector_data/
/backend/vector_data/

# 不要统一忽略 *.sql：
# 数据库建表脚本和迁移脚本通常应该提交
# 数据库导出文件应放进 backups/ 等忽略目录


# ==================================================
# 8. 日志、进程与临时文件
# ==================================================

logs/
*.log
*.pid
*.pid.lock

*.tmp
*.temp
*.bak
*.swp
*.swo
*~


# ==================================================
# 9. 编辑器与操作系统生成物
# ==================================================

.vscode/
.idea/
*.iml
*.suo
*.user

.DS_Store
._*
Thumbs.db
ehthumbs.db
Desktop.ini
$RECYCLE.BIN/

# Jupyter自动检查点
.ipynb_checkpoints/
```

## 阶段二：SDD 开发流程

### 1. 需求分析：PRD、用户故事和验收标准

目标：

根据 README.md 当前的内容和 AI 交流得到下面两个文档

- 产品需求文档（PRD.md）：包含产品目标、用户、功能范围、业务规则和非功能需求。
- 用户故事与验收标准（user-stories.md）：每条用户故事附上可验证的验收标准。

完成后要根据用户故事再一次修改 PRD.md

将两个文件放在项目根目录的 /docs/spec/MVP 下

### 2. 原型图设计

目标：

在写代码之前，先用 HTML 画出页面的交互原型，确认最终的效果和交互过程。

>交互提示词示例：

```text
请阅读 README.md、docs/spec/MVP 下的 PRD.md 和 user-stories.md，为当前 MVP 版本设计可交互的页面原型。

要求：
1. 根据需求和用户故事梳理页面、功能入口及主要操作流程，覆盖当前 MVP 范围，不自行增加功能。
2. 使用 HTML、CSS、JavaScript 实现原型，采用虚构数据，暂不连接后端、数据库或真实模型。
3. 主要按钮、页面切换、表单和弹窗可以操作，让我能体验从进入页面到完成任务的完整流程。
4. 按页面需要展示正常、加载、空数据、失败、提交中等状态；仅在需求涉及账号和权限时增加无权限、登录过期状态。
5. 页面清晰易用，布局和组件风格统一。
6. 将原型文件放在 docs/spec/MVP/prototype/，以 index.html 作为统一入口，确保双击即可打开操作，并检查主要交互是否正常。
7. 简要说明页面与用户故事的对应关系，以及需要我体验确认的重点。

信息不足时，只询问影响主要流程或布局的关键问题，其余采用合理默认方案并说明。
```

生成初始原型图后，用 VibeCoding 的方式改造原型图。

### 3. 业务设计：把需求和原型变成技术方案

目标：

将需求和原型转化为可执行的技术方案。

产出：

1. 项目宪法（根目录下的 AGENTS.md 还有MVP下的 AGENTS.md）

1. 业务术语表（glossary.md）

1. 数据库设计（db_design.md）

1. 接口设计（api_design.md）

#### 3.1 编写项目宪法（AGENTS.md）

##### 3.1.1 根目录下的 AGENTS.md

>交互提示词示例：

```text
请为"项目名"项目生成根目录下的 AGENTS.md，内容包括：

技术栈：参考 README.md 里的技术栈

目录结构：参考 README.md 里的目录结构

业务接口响应格式：统一为 {code, message, data}。初始化健康检查 /api/health 为明确例外，返回 {"status":"ok"}。

请生成完整的 AGENTS.md 文件，放在项目根目录下
```

##### 3.1.2 MVP目录下的 AGENTS.md

>交互提示词示例：

```text
请为"项目名"项目生成 MVP 版本的 AGENTS.md，内容包括：

约束条件：……

代码风格：……

验收相关约束：……

MVP 版本全部产出文档清单：用来存放 docs/spec/MVP/ 下的所有文档

请生成完整的 AGENTS.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.2 业务术语表（Glossary）

>交互提示词示例：

```text
请根据需求文档和用户故事生成业务术语表。

格式：术语 | 定义

请生成完整的 glossary.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.3 数据库设计

>交互提示词示例：

```text
请根据 docs/spec/MVP 下的现有文档，设计"项目名"的数据库结构。

请生成完整的数据库设计文档，包含：
- 每张表的字段、类型、约束、描述
- 如果还需要其他表，你要提示我
- 明确主键、外键、表之间的关系、必要索引、数据归属和删除规则

生成完整的 db_design.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.4 API接口设计

>交互提示词示例：

```text
请根据 docs/spec/MVP 下的现有文档，生成"项目名"当前 MVP 范围内的接口

需要注意的是：
1. 接口名不要太复杂，简单一些即可
2. 仅设计当前 MVP 范围内的接口，覆盖原型中的实际操作，不要遗漏
3. 明确请求方法、路径、参数及校验、成功与错误响应、HTTP 状态码、业务 code、认证方式和数据访问权限

请生成完整的 api_design.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.5 添加文档清单

将 MVP 版本产出的文档清单添加到 /docs/spec/MVP/AGENTS.md 的“MVP 版本全部产出文档清单”中，应该包括：

- PRD.md
- user-stories.md
- prototype/
- glossary.md
- db_design.md
- api_design.md
- SDD-开发单元拆解.md # 第 4 部分生成

### 4. 编码

#### 4.1 SDD开发单元拆解

>交互提示词示例：

```text
请阅读项目根目录下的 AGENTS.md 以及 docs/spec/MVP 下的所有文档，同时检查现有代码和已完成的开发单元；项目尚未初始化时，按新项目处理。把 MVP 阶段拆成适合在 SDD 开发中逐次交给 AI 完成的开发单元。

一个开发单元必须围绕一个明确目标，按本开发单元的需要，在一次开发中完成涉及的前端、后端、数据库、AI 调用、权限处理、异常状态和测试，并且完成后可以独立运行、演示和验收。业务开发单元应形成用户操作到结果展示的完整闭环；基础能力单元应形成自身的实现、测试和验收闭环。

不要只按页面或 API 拆分，也不要把多个无关功能合并。如果一个单元过大，请继续拆分；如果多个小任务必须一起才能形成完整功能，请合并。请按照前置依赖安排顺序。

输出以下内容：

1. 开发单元编号和名称；
2. 主要目标；
3. 前置依赖；
4. 本次包含的范围；
5. 本次明确不包含的范围；
6. 预计涉及的页面、接口和数据；
7. 验收标准；
8. 测试方式；
9. 优先级和复杂度。

请使用具体名称，不要使用“功能一”“功能二”等占位名称，并指出哪些内容属于基础能力、业务功能或公共能力。

生成 SDD-开发单元拆解.md 文件放在项目根目录的 /docs/spec/MVP 下
```

#### 4.2 为开发单元生成执行提示词

>交互提示词示例：

```text
请阅读项目根目录下的 AGENTS.md、docs/spec/MVP 下的所有文档，同时检查现有代码和已完成的开发单元；项目尚未初始化时，按新项目处理。为指定的开发单元生成一份可以直接交给编程 AI 执行的开发提示词。

指定开发单元：
【填写开发单元编号和名称】

生成的提示词必须包含：

1. 开发目标；
2. 本次必须完成的内容；
3. 本次明确不处理的内容；
4. 需要先阅读的项目文件；
5. 涉及的前端、后端、数据库和 AI 服务；
6. 接口、数据保存和权限要求；
7. 加载、空数据、失败、超时和重复提交处理；
8. 测试要求：要求执行者将测试代码放在项目根目录的 tests/MVP/ 下，运行相关测试并修复问题；
9. 验收标准：明确本开发单元满足哪些可验证的条件才算完成；
10. 汇报要求：要求执行者在完成后汇报实现内容、测试命令与实际结果、验收情况，以及未完成或未验证的事项。

请严格限制在指定开发单元范围内。如果发现文档冲突、前置依赖未完成或需求信息不足，请在提示词中明确列出，不要擅自扩大范围。

每个开发单元都应尽量形成完整闭环：用户操作、前端交互、后端处理、数据保存、结果展示、异常处理和测试。
只生成开发提示词，不要直接写代码。
```

#### 4.3 实现开发单元

>交互提示词示例：

```text
请严格按照下面的开发任务提示词完成本开发单元：

【本单元的执行提示词】
```

### 5. MVP整体联调、回归测试与验收

目标：

验证各开发单元组合后是否按预期工作，确保当前版本符合需求文档、用户故事和验收标准。

>交互提示词示例：

```text
请结合当前代码、已有测试和 docs/spec/MVP 下的文档，对当前版本进行整体联调、回归测试与验收。

运行已有测试，并补充缺失的跨功能流程测试；
新增测试代码放在项目根目录的 tests/MVP/ 下。

使用独立测试环境或测试数据库，只清理本次测试创建的数据。

修复发现的问题，并重新运行受影响的测试。
对未实现的需求、文档冲突和无法验证的事项，列出清单及原因；
不要自行增加 MVP 范围之外的功能。

最后汇报测试命令、实际结果，以及各项验收标准的满足情况。
```

### 6. 本地运行

目标：

验证项目能够按文档在本地运行。

>交互提示词示例：

```text
请根据项目实际配置，完善 README.md 中的运行说明，包括：

1. 依赖安装；
2. 环境变量配置；
3. 数据库初始化；
4. 本地前后端启动命令、访问地址和停止方式；

实际验证本地启动流程。
如发现启动问题，请排查并修复。
将需要长期遵守的开发注意事项补充到 AGENTS.md，
将面向使用者的运行说明补充到 README.md。
```

## 阶段三: 服务器部署与上线